# 5. UMAP: aproximación de variedades y proyección

Estudiaremos su grafo de vecindad, las escalas locales y el objetivo de optimización. La formulación topológica motiva un algoritmo práctico; sus supuestos no deben confundirse con garantías de preservación exacta de los datos.



*Fig. 5.1. Conservar el grafo aproximadamente no garantiza toda la geometría.*

## 5.1. Proyección no lineal

Los datos con muchos atributos pueden concentrarse cerca de estructuras de menor dimensión, pero esto es una hipótesis que debe evaluarse, no una propiedad de todos los datasets. La maldición de la dimensionalidad depende del muestreo, la distribución y la métrica; no implica que cualquier volumen de radio fijo crezca con la dimensión.

PCA busca un subespacio de máxima varianza. t-SNE y UMAP optimizan representaciones no lineales basadas en vecindades. UMAP construye un grafo ponderado y busca un embedding que reproduzca sus relaciones. Puede revelar estructura a distintas escalas, pero **no garantiza conservar las distancias globales, densidades o toda la topología**.

El costo depende de la búsqueda de vecinos, número de épocas, dimensionalidad, métrica y hardware. La búsqueda aproximada y el muestreo negativo favorecen su escalabilidad; $O(N\log N)$ no es una cota universal para cualquier configuración. t-SNE también dispone de aproximaciones que evitan el costo cuadrático del método exacto.



*Fig. 5.2. La elección depende del objetivo y de la validación.*



*Fig. 5.3. El preprocesamiento puede cambiar por completo el grafo de vecinos.*

### **5.1.1. Las Tres Asunciones Topológicas de UMAP**



*Fig. 5.4. Los datos reales no vienen con una garantía de cumplir estos supuestos.*

La motivación geométrica de UMAP utiliza una distribución uniforme respecto de una métrica riemanniana, una métrica aproximadamente constante en vecindarios pequeños y conectividad local. Los datos reales pueden desviarse de estas hipótesis. La implementación adapta escalas y construye un grafo; no estima de forma completa una variedad continua ni certifica sus supuestos.

#### 5.1.1.1. Uniformidad respecto de una métrica

«Uniforme» se refiere a una medida de volumen asociada a la métrica, no necesariamente a densidad uniforme en las coordenadas observadas. UMAP ajusta una escala local para aproximar ese planteamiento sobre una muestra finita.



*Fig. 5.5. Uniformidad métrica no exige densidad uniforme en el espacio observado.*



*Fig. 5.6. Los pesos difusos no están normalizados como probabilidades por fila.*

Un tensor métrico riemanniano define productos internos locales: permite medir longitudes y ángulos de vectores tangentes. En coordenadas se representa mediante una matriz **simétrica definida positiva** que puede variar con la posición. Cambiar coordenadas también puede hacer variable esa matriz incluso en un espacio plano; una métrica variable no prueba por sí sola curvatura.



*Fig. 5.7. El ajuste local no recupera un tensor métrico completo.*



*Fig. 5.8. La forma cuadrática da longitudes infinitesimales positivas.*

##### 1. Coordenadas cartesianas ortonormales

En el plano euclidiano con la misma unidad en ambos ejes,

$$ds^2=dx^2+dy^2.$$

La matriz métrica es la identidad. La ausencia de término mixto expresa ortogonalidad de los vectores de la base coordenada.



*Fig. 5.9. ds² = dx² + dy² en coordenadas cartesianas ortonormales.*

##### 2. Escala y ángulo de los ejes

Si un paso en $x$ mide 1.6 unidades y uno en $y$ mide una unidad, manteniendo ortogonalidad,

$$ds^2=2.56\,dx^2+dy^2.$$

Con vectores base unitarios que forman un ángulo $\theta$, la norma del desplazamiento suma

$$ds^2=dx^2+dy^2+2\cos(\theta)\,dx\,dy.$$

El signo positivo proviene del producto interno de la **suma** de los desplazamientos. Esta es una base oblicua, no una prueba de que el espacio sea curvo. Si se usa la conversión real de millas a kilómetros, el factor lineal es 1.609344, no exactamente 1.6.



*Fig. 5.10. Un término mixto expresa el producto interno de las direcciones.*

##### 3. Forma cuadrática local

$$ds^2=g_{11}dx^2+2g_{12}dx\,dy+g_{22}dy^2=d\mathbf{x}^Tg\,d\mathbf{x}.$$

Con $g_{12}=g_{21}$, la matriz debe ser definida positiva para que todo vector tangente no nulo tenga longitud positiva. En 2D basta exigir $g_{11}>0$ y $\det g>0$. Los términos mixtos describen productos internos entre direcciones coordenadas, no interferencia física.



*Fig. 5.11. Una matriz variable no implica necesariamente curvatura intrínseca.*

##### 4. Dependencia de la posición

En una variedad, $g=g(x)$ describe la medición local. También puede variar por usar coordenadas no cartesianas en un espacio plano. Para obtener una distancia de camino se integra su longitud:

$$\ell(\gamma)=\int\sqrt{\dot\gamma(t)^T g(\gamma(t))\dot\gamma(t)}\,dt.$$

La distancia geodésica es el ínfimo de estas longitudes entre los puntos. Una forma cuadrática en un único punto da una longitud infinitesimal, no automáticamente la distancia exacta entre dos puntos alejados.



*Fig. 5.12. La distancia geodésica minimiza longitudes de caminos.*



*Fig. 5.13. La aproximación local debe justificarse a la escala usada.*

##### 5. Relación con la escala de UMAP

La idea de reescalar localmente puede ilustrarse como $ds\approx\|d\mathbf{x}\|/\sigma_i$. Es una interpretación didáctica, **no una recuperación de todo el tensor métrico de los datos**.

El algoritmo calcula una escala $\sigma_i$ sobre vecinos y un desplazamiento $\rho_i$. Para $k$ vecinos, la calibración busca aproximadamente

$$\sum_{j\in\mathcal N_k(i)}\exp\left[-\frac{\max(0,d(x_i,x_j)-\rho_i)}{\sigma_i}\right]=\log_2 k.$$

La métrica de entrada es configurable y no tiene que ser euclídea. Los pesos resultantes expresan pertenencia difusa; no son probabilidades normalizadas por fila.



*Fig. 5.14. UMAP admite distintas métricas de entrada; no solo distancia euclídea.*

#### **5.1.1.2. Cimiento 2: Métrica Localmente Constante**



*Fig. 5.15. Un plano tangente aproxima la geometría a pequeña escala.*

Una métrica suave puede aproximarse como constante en un vecindario suficientemente pequeño. El espacio tangente es una aproximación lineal local; esto no vuelve plana una región finita ni elimina su curvatura. La calidad de la aproximación depende de la escala y del muestreo.

El algoritmo utiliza distancias de entrada y escalas locales como aproximación práctica de relaciones de vecindad. No calcula geodésicas exactas entre todos los pares ni resuelve las ecuaciones de una variedad reconstruida. La aproximación euclídea local solo es apropiada cuando la representación, métrica y tamaño del vecindario la justifican.



*Fig. 5.16. La resta de ρᵢ y la escala σᵢ cumplen papeles diferentes.*

##### **1. Analogía: El Agrimensor y la Curvatura de la Tierra**



*Fig. 5.17. Una aproximación local no justifica usarla a cualquier distancia.*

Una pequeña parcela puede aproximarse con un plano aunque la superficie terrestre sea curva. Para trayectos largos puede ser necesario considerar la curvatura. En datos, la analogía recuerda que una aproximación local no se extiende automáticamente a todas las distancias del conjunto.

#### **5.1.1.3. Cimiento 3: Conexidad Local (Local Connectivity)**

##### 1. Conectividad local y global

Ser localmente conexo significa disponer de una base de vecindarios conexos en torno a cada punto. No implica que todo el espacio sea conexo: dos componentes separadas pueden ser localmente conexas. Tampoco exige que un grafo finito de vecinos tenga una sola componente. Las distancias en alta dimensión no se vuelven necesariamente enormes; su escala depende de la representación y de la métrica.

Con `local_connectivity=1`, y en el caso habitual de distancias positivas finitas, $\rho_i$ corresponde a la distancia al vecino más próximo distinto. Se define el peso dirigido

$$w_{i\to j}=\exp\left[-\frac{\max(0,d(x_i,x_j)-\rho_i)}{\sigma_i}\right]$$

para vecinos seleccionados y cero fuera del grafo. Duplicados, otras configuraciones de conectividad local y umbrales de desconexión requieren considerar las reglas de la implementación. No se debe identificar siempre $\rho_i$ con un mínimo sobre todas las muestras sin esas condiciones.



*Fig. 5.18. Un enlace al vecino próximo no une necesariamente todas las componentes.*

Para un vecino cuya distancia satisface $d(x_i,x_j)\leq\rho_i$, el peso es $e^0=1$. Esto da un enlace local fuerte en el caso habitual. **No garantiza conectar todas las componentes** del grafo ni conservar ese enlace con distancia cero en el embedding final.

##### 2. Analogía de las cuerdas

Si cada montañista se ata a su compañero más cercano, puede no quedar nadie aislado individualmente y aun así formarse varios grupos desconectados. La analogía ilustra por qué un enlace local fuerte no crea necesariamente una red continua que abarque a todos.



*Fig. 5.19. Puede no haber puntos aislados y seguir habiendo grupos desconectados.*

Restar $ho_i$ permite asignar peso 1 a los enlaces suficientemente próximos según la escala local. Es una decisión de construcción del grafo; no certifica una variedad conectada, ausencia de ruido o fidelidad topológica de la proyección.

## **5.2. La Costura Algebraica: Unión Difusa (Fuzzy Union)**

Al combinar estos tres cimientos,  **UMAP construye un espacio métrico local adaptativo para cada punto de datos.** Sin embargo, dado que los  **parámetros de escala métrica varían localmente** ($\sigma_i \neq \sigma_j$),  **la fuerza de enlace condicional del punto $i$ al punto $j$ no coincide con la del sentido inverso** ($p_{j \mid i} \neq p_{i \mid j}$).



*Fig. 5.20. Una relación de vecindad dirigida no tiene por qué ser recíproca.*

Las escalas locales pueden ser distintas, así que $w_{i	o j}$ y $w_{j	o i}$ pueden diferir. t-SNE combina probabilidades condicionales normalizadas y UMAP combina pertenencias difusas. Comparar sus números directamente como si fueran la misma magnitud, o afirmar que el promedio necesariamente rompe puentes, sería incorrecto.



*Fig. 5.21. El promedio de t-SNE no prueba por sí mismo que se rompan puentes.*

La unión difusa estándar combina pesos dirigidos mediante

$$w_{ij}=w_{i\to j}+w_{j\to i}-w_{i\to j}w_{j\to i}.$$

Si los pesos son 0.2 y 0.8, el resultado es 0.84. Es una t-conorma con forma de OR probabilístico, pero los pesos son pertenencias difusas y no se está suponiendo una independencia probabilística real entre observaciones. El resultado es simétrico y no menor que cada entrada. `set_op_mix_ratio` permite otras mezclas de unión e intersección.



*Fig. 5.22. wᵢⱼ = u + v − uv; con 0,2 y 0,8 el resultado es 0,84.*

Si un sentido tiene peso 1, la unión tiene peso 1. Esto conserva ese enlace en el grafo simétrico. No garantiza conservar todos los puentes existentes en una variedad desconocida ni las posiciones globales del embedding.



*Fig. 5.23. Preservar una arista del grafo no garantiza una geometría global exacta.*



*Fig. 5.24. La construcción del grafo y la optimización son etapas diferentes.*

## **5.3. Geometría en Baja Dimensión y la Optimización del Espacio Latente**

> Una vez  **construida la estructura topológica difusa de alta dimensión** (representada por la matriz de pesos $P$), el objetivo es  **encontrar una configuración de puntos correspondientes en baja dimensión $\mathbf{y}_1, \dots, \mathbf{y}_n \in \mathbb{R}^d$ cuya estructura topológica difusa $Q$ sea lo más idéntica posible a $P$.**



*Fig. 5.25. La baja dimensión obliga a compromisos entre aristas y no aristas.*

### 5.3.1. Afinidad en baja dimensión

Con distancia euclídea de salida, UMAP usa una curva no normalizada sobre pares:

$$q_{ij}=\frac{1}{1+a\|y_i-y_j\|^{2b}},\qquad a,b>0.$$

Es una pertenencia difusa en $(0,1]$, no una distribución cuya suma total sea 1. La elección del espacio de salida es una configuración; bajar a 2D no elimina por sí solo las diferencias de densidad.



*Fig. 5.26. q = 1 / (1 + a·d²ᵇ); no se normaliza sobre todos los pares.*

Por defecto, $a$ y $b$ se ajustan para aproximar una curva objetivo gobernada por `min_dist` y `spread`:

$$\psi(d)=\begin{cases}1,&d\leq\text{min_dist},\\\exp[-(d-\text{min_dist})/\text{spread}],&d>\text{min_dist}.\end{cases}$$

`min_dist` controla una preferencia de empaquetamiento, **no impone una distancia mínima rígida entre todos los puntos**. `spread` fija una escala de referencia. Cambiar estos parámetros afecta la forma del mapa y no recupera automáticamente la densidad original.



*Fig. 5.27. min_dist es una preferencia de empaquetamiento, no una restricción rígida.*



*Fig. 5.28. Las curvas ilustran parámetros; no son densidades normalizadas.*

Con $a=b=1$, el núcleo $1/(1+d^2)$ coincide con la forma no normalizada usada en t-SNE. El objetivo y la normalización son distintos, por lo que los dos algoritmos no se vuelven equivalentes.



*Fig. 5.29. El parámetro a modifica la escala del decaimiento en conjunto con b.*



*Fig. 5.30. a y b se ajustan conjuntamente; no son controles independientes exactos.*

Un `min_dist` pequeño permite grupos más compactos y uno mayor suele favorecer dispersión. Es una tendencia de la optimización, no una garantía exacta sobre cada grupo. Los parámetros $a,b$ se ajustan conjuntamente: no deben interpretarse como controles completamente independientes de ancho y pendiente.



*Fig. 5.31. La densidad del mapa no estima necesariamente la densidad original.*

### 5.3.2. Entropía cruzada de conjuntos difusos

Una expresión teórica utilizada para comparar las pertenencias es

$$\mathcal L=\sum_{i<j}\left[w_{ij}\log\frac{w_{ij}}{q_{ij}}+(1-w_{ij})\log\frac{1-w_{ij}}{1-q_{ij}}\right].$$

Para $P$ fijo, los términos que dependen solo de $w_{ij}$ son constantes; minimizar esto equivale a minimizar la entropía cruzada binaria correspondiente. Se usa la convención de límite $0\log0=0$. La implementación práctica muestrea aristas y ejemplos negativos; no calcula ingenuamente todos los pares en cada paso ni reproduce sin matices el objetivo denso completo.



*Fig. 5.32. El entrenamiento práctico no evalúa todos los pares en cada actualización.*

#### 5.3.2.1. Atracción

Si $w_{ij}$ está cerca de 1, el término relevante favorece un $q_{ij}$ alto, acercando las coordenadas. La optimización equilibra muchas relaciones incompatibles en un espacio pequeño. No conserva de manera perfecta todas las vecindades.



*Fig. 5.33. El embedding no reproduce perfectamente todas las vecindades.*

**Repulsión.** Para pesos cercanos a cero, la comparación favorece afinidades bajas. En la implementación la repulsión se aproxima con muestreo negativo y depende de distancias y parámetros; no es una fuerza constante sobre todos los pares. t-SNE también contiene repulsión mediante la normalización de sus afinidades de salida. Ninguno garantiza distancias globales exactas ni una disposición relativa de grupos con significado métrico universal.



*Fig. 5.34. La fuerza depende de distancia y parámetros; no es constante.*



*Fig. 5.35. UMAP y t-SNE no garantizan distancias globales calibradas.*

## **5.4. ¿Cuándo usar t-SNE y cuándo usar UMAP?**

### 5.4.1. Elegir según el objetivo

Ambos métodos son útiles para explorar representaciones. La hipótesis de variedad motiva su diseño, pero no prueba que un dataset la cumpla ni que cada isla corresponda a una clase. Conviene contrastar mapas con la estructura de vecinos original y con el objetivo del análisis.

### 5.4.2. t-SNE: énfasis en vecindades

#### 5.4.2.1. Fortalezas

Su objetivo prioriza afinidades locales y sus colas pesadas alivian el hacinamiento. Puede revelar grupos y transiciones a distintas escalas. No garantiza microgrupos perfectos; la penalización solo diverge en el límite cuando una afinidad de salida tiende a cero para un par con peso positivo.

#### 5.4.2.2. Límites y comparación con UMAP

| Pregunta | t-SNE | UMAP |
|---|---|---|
| ¿Qué relaciones prioriza? | Afinidades locales con escalas adaptativas | Grafo de vecinos y sus pesos difusos |
| ¿Distancias entre islas calibradas? | No garantizadas | No garantizadas |
| ¿Proyectar observaciones nuevas? | `sklearn.manifold.TSNE` no ofrece `transform`; otras implementaciones y variantes paramétricas sí permiten extensión | `umap-learn` ofrece `transform` bajo condiciones compatibles |
| ¿Escalabilidad? | Exacto cuadrático; Barnes–Hut y otros métodos aproximados reducen el costo | Búsqueda de vecinos y optimización muestreada; depende de parámetros y datos |
| ¿Usar como feature engineering? | El TSNE estándar de scikit-learn se orienta a visualización | Puede formar parte de un pipeline ajustado solo con entrenamiento y validado fuera de muestra |

La orientación y la escala de un mapa no tienen significado intrínseco. UMAP puede producir clusters aparentes y distorsionar densidades; tampoco sustituye una validación de clustering. Al usar etiquetas en variantes supervisadas, hay que evitar filtrarlas desde test. Para generalizar un mapa, conviene considerar además cambios de distribución y estabilidad entre semillas.

## Referencias y lecturas

- [UMAP · McInnes, Healy y Melville](https://arxiv.org/abs/1802.03426)
- [Cómo funciona UMAP](https://umap-learn.readthedocs.io/en/latest/how_umap_works.html)
- [API de umap-learn](https://umap-learn.readthedocs.io/en/latest/api.html)
- [TSNE · scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

Referencias del material de origen: [tesis de grado](https://drive.google.com/file/d/1fxbhCJ6f9nTZXvzaqboVLrMFQE1AlXZv/view) y [Dive into Deep Learning](https://d2l.ai/). Los textos y figuras de terceros conservan sus créditos y condiciones.